In [ ]:
MODEL_NAME = "single_head_single_repetition_pattern_low_dim_masked_first_repetition"

In [ ]:
VOCABULARY = [c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"]
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 16
NUM_HEADS = 1

import copy_transformer.training
import transformer_lens
import infra.dataset_configs as dataset_configs
import copy_transformer.tokenizer
from pathlib import Path

training_dataset_config = dataset_configs.UniqueTokenPatternConfig(
    vocabulary=VOCABULARY,
    max_pattern_length=CONTEXT_LENGTH // 2,
    min_pattern_length=2,
    iterable=False,
    length=10_000,
    mask_first_repetition=True,
)

training_config = copy_transformer.training.TrainingConfig(
    model_name=MODEL_NAME,
    epochs=10,
    dataset_config=training_dataset_config,
    validation_dataset_config=training_dataset_config,
)

tokenizer = copy_transformer.tokenizer.SingleCharTokenizer(
    alphabet=VOCABULARY,
    bos_token=">",
    eos_token="<",
    unk_token="?",
    pad_token="_",
)

model_config = transformer_lens.HookedTransformerConfig(
    d_model=EMBEDDING_DIM,
    n_heads=NUM_HEADS,
    d_head=EMBEDDING_DIM // NUM_HEADS,
    n_layers=2,
    n_ctx=CONTEXT_LENGTH,
    attn_only=True,
    d_vocab=tokenizer.vocab_size,
)

model = copy_transformer.training.train_transformer(
    config=training_config,
    model_config=model_config,
    tokenizer=tokenizer,
)

In [ ]:
EXPERIMENT_NAME = MODEL_NAME + "_" + ...
ACT_SITES = ["blocks.0.hook_resid_post", "blocks.1.hook_resid_post"]

import subspace_partition.subspace_partition
import infra.dataset_configs
import subspace_partition.model_configs
from pathlib import Path

_, model_training_config = subspace_partition.model_configs.load_model(
    MODEL_NAME, load_training_config=True
)

subspace_partition_dataset_config = model_training_config.dataset_config
subspace_partition_dataset_config.iterable = True
subspace_partition_dataset_config.length = "infinite"

subspace_partition_config = (
    subspace_partition.subspace_partition.SubspacePartitionConfig(
        exp_name=EXPERIMENT_NAME,
        model_name=MODEL_NAME,
        dataset_config=subspace_partition_dataset_config,
        act_sites=ACT_SITES,
        unit_size=2,  # Must divide EMBEDDING_DIM evenly
        max_steps=20_000,
        merge_start=2_000,
        merge_interval=2_000,
        output_dir=Path("out/subspace_partition"),
        search_steps=1,
    )
)

subspace_partition.subspace_partition.run_subspace_partition(
    cfg=subspace_partition_config
)

In [ ]:
import subspace_partition.preimage.cache_act
import subspace_partition.model_configs
import infra.dataset_configs

_, model_training_config = subspace_partition.model_configs.load_model(
    MODEL_NAME, load_training_config=True
)

cached_act_dataset_config = model_training_config.dataset_config
cached_act_dataset_config.length = 10_000

subspace_partition.preimage.cache_act.run_cache_act(
    model_name=MODEL_NAME,
    dataset_config=cached_act_dataset_config,
    act_sites=ACT_SITES,
)

In [ ]:
import subspace_partition.preimage.build_index

subspace_partition.preimage.build_index.run_build_index(
    experiment_name=EXPERIMENT_NAME,
)

In [ ]:
import os
os.environ["INDEX_NAME"] = f"index-{EXPERIMENT_NAME}-cosine"
! ./start_streamlit_app.sh "../out/index/$INDEX_NAME"